# CatBoost-метауровень с траекториями статусов

CPU. Сравниваем два CatBoost depth 4 с одинаковыми обучающими строками, параметрами, базовыми прогнозами и объёмом Session6. Второй получает дополнительные признаки о движении груза по статусам и форме прогнозов на соседних горизонтах. Обе модели учат поправку к прогнозам баз. Проверка идёт на двух предтестовых неделях, затем на локальном test. Рецепт задан заранее. Известные факты после стартов прогноза не используются как признаки.

Новый кандидат допускается, если улучшает средний дневной score на обеих неделях и не менее чем на 0.0003 на локальном test, без ухудшения общего score на test. Октябрьский test уже изучался, оценка получена до финального переобучения. Если порог пройден, новый метауровень переобучается на всех 42 днях честных исторических прогнозов и используется с базами после полного refit. Родительский конкурсный CSV задаёт объёмы по горизонтам.

На сервере входные прогнозы исходного бленда ожидаются в output/blend_submit_948a52023607. Если они в другой папке, задайте BLEND_BASE_RUN. Для локальной проверки на архиве укажите TEST_PATH и OOF_PATH явно. Финальный прогноз требует согласованную пару train_val и full_history от исходного запуска.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_status_meta_cpu'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)

DATA = ROOT/'blend_inputs/session6'
BASE_RUN = Path(os.environ.get('BLEND_BASE_RUN',str(ROOT/'output/blend_submit_948a52023607')))
PARENT = Path(os.environ.get('STATUS_META_PARENT',str(ROOT/'status_meta_parent.csv')))
RUN_TEST = BASE_RUN/'train_val/test_predictions.parquet'
RUN_FUTURE = BASE_RUN/'full_history/future_predictions.parquet'
SIGNALS = ['target_1h']+[f'status_{i}' for i in range(1,7)]
STEP = pd.Timedelta(minutes=30)
MIN_TEST_GAIN = .0003


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
def load_statuses(path):
    frame = pd.read_parquet(path,columns=['timestamp','route_id',*SIGNALS]).sort_values(['timestamp','route_id'])
    times = pd.DatetimeIndex(frame.timestamp.unique())
    routes = pd.Index(np.sort(frame.route_id.unique()))
    if (len(frame)!=len(times)*len(routes) or frame.duplicated(['timestamp','route_id']).any()
            or not times.equals(pd.date_range(times[0],times[-1],freq=STEP))):
        raise ValueError('Неполная история по маршрутам')
    values = frame[SIGNALS].to_numpy(dtype=np.float32).reshape(len(times),len(routes),len(SIGNALS))
    if not np.isfinite(values).all() or np.any(values<0):
        raise ValueError('Некорректные значения статусов')
    network = values.sum(axis=1,dtype=np.float64)[:,None,:]
    values = np.divide(values,network,out=np.zeros_like(values),where=network>0)
    return times,routes,np.log1p(1000*values)

def status_features(frame):
    ti = TIMES_STATUS.get_indexer(pd.DatetimeIndex(frame.cutoff))
    ri = ROUTES_STATUS.get_indexer(frame.route_id)
    if np.any(ti<336) or np.any(ri<0):
        raise ValueError('Нет причинной истории для признаков')
    now = LOG_SHARE[ti,ri]
    prev_1h = LOG_SHARE[ti-2,ri]
    prev_2h = LOG_SHARE[ti-4,ri]
    day = LOG_SHARE[ti-48,ri]
    week = LOG_SHARE[ti-336,ri]
    typical = (day+week)/2
    pulse = now-typical
    return np.column_stack([now,now-prev_1h,prev_1h-prev_2h,pulse,
                            pulse[:,1:4]-pulse[:,[0]],pulse[:,4:7]-pulse[:,[0]]])

def horizon_features(frame,matrix,level):
    series = pd.DataFrame(matrix,columns=BASES,index=pd.MultiIndex.from_frame(frame[KEYS]))
    before = frame[KEYS].copy()
    after = frame[KEYS].copy()
    before['horizon'] = (before.horizon-1).clip(lower=1)
    after['horizon'] = (after.horizon+1).clip(upper=8)
    left = series.reindex(pd.MultiIndex.from_frame(before)).to_numpy()
    right = series.reindex(pd.MultiIndex.from_frame(after)).to_numpy()
    if not np.isfinite(left).all() or not np.isfinite(right).all():
        raise ValueError('Не хватает прогнозов соседнего горизонта')
    return np.column_stack([(matrix-left)/level[:,None],(right-matrix)/level[:,None]])

def design(frame,matrix,baseline,augmented):
    context,level = meta_features(frame,matrix,baseline)
    if augmented:
        context = np.column_stack([context,status_features(frame),horizon_features(frame,matrix,level)])
    return np.clip(context,-50,50).astype(np.float32,copy=False),level

def train_pair(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix,frame.y)
    baseline = matrix @ weights
    models = {}
    for name,augmented in [('control',False),('status_meta',True)]:
        features,level = design(frame,matrix,baseline,augmented)
        model = CatBoostRegressor(depth=4,iterations=220,loss_function='MAE',learning_rate=.035,
                                  l2_leaf_reg=20,random_seed=SEED,verbose=False,allow_writing_files=False,
                                  task_type='CPU',thread_count=4)
        model.fit(features,(frame.y.to_numpy(dtype=float)-baseline)/level,sample_weight=level/level.mean())
        models[name] = model
    return weights,models

def predict_pair(frame,fitted):
    weights,models = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    result = {}
    for name,model in models.items():
        features,level = design(frame,matrix,baseline,name=='status_meta')
        residual = np.clip(model.predict(features),-.5,.5)*level
        result[name] = with_volume(frame,baseline+residual)
    return result

def summarize(frame,forecasts,fold,period):
    rows = []
    for name,prediction in forecasts.items():
        daily = [metric(g.y,prediction[g.index])['score'] for _,g in frame.groupby('cutoff')]
        rows.append(dict(fold=fold,period=period,variant=name,**metric(frame.y,prediction),
                         daily_mean=float(np.mean(daily)),daily_p90=float(np.quantile(daily,.9)),
                         daily_worst=float(np.max(daily))))
    return rows


In [ ]:
TIMES_STATUS,ROUTES_STATUS,LOG_SHARE = load_statuses(DATA/'train_solo_track.parquet')
oof = read_panel(OOF_PATH,'base_oof.parquet','2025-09-20','2025-10-18')
rows = []
for start,end in [('2025-10-04','2025-10-11'),('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff+oof.horizon*STEP < pd.Timestamp(start)].reset_index(drop=True)
    val = oof.loc[(oof.cutoff>=start)&(oof.cutoff<end)].reset_index(drop=True)
    print('Подбор',start,flush=True)
    fitted = train_pair(train)
    rows.extend(summarize(train,predict_pair(train,fitted),start,'train_oof'))
    rows.extend(summarize(val,predict_pair(val,fitted),start,'val'))
    display(pd.DataFrame(rows).loc[lambda table: table.fold.eq(start)])
validation = pd.DataFrame(rows)
validation.to_csv(OUT/'validation_metrics.csv',index=False)
ranking = validation.loc[validation.period.eq('val')].pivot(index='fold',columns='variant',values='daily_mean')
passed_validation = bool(ranking.status_meta.lt(ranking.control).all())
(OUT/'frozen_recipe.json').write_text(json.dumps(dict(passed_validation=passed_validation,min_test_gain=MIN_TEST_GAIN,test_used_for_tuning=False),indent=2))


In [ ]:
test_input = TEST_PATH if TEST_PATH is not None else RUN_TEST if RUN_TEST.is_file() else None
test = read_panel(test_input,'test_predictions.parquet','2025-10-18','2025-11-01')
print('Локальный test',flush=True)
fitted = train_pair(oof)
rows.extend(summarize(oof,predict_pair(oof,fitted),'2025-10-18','train_oof'))
test_forecasts = predict_pair(test,fitted)
rows.extend(summarize(test,test_forecasts,'2025-10-18','test'))
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
display(metrics.loc[metrics.fold.eq('2025-10-18')])
evaluation = metrics.loc[metrics.period.eq('test')].set_index('variant')
gain = float(evaluation.loc['control','daily_mean']-evaluation.loc['status_meta','daily_mean'])
promoted = bool(passed_validation and gain>=MIN_TEST_GAIN
                and evaluation.loc['status_meta','score']<=evaluation.loc['control','score'])
(OUT/'decision.json').write_text(json.dumps(dict(promoted=promoted,passed_validation=passed_validation,
                                                 test_daily_gain=gain,test_before_full_refit=True),indent=2))
print('Прошёл порог для последней попытки:',promoted,flush=True)


In [ ]:
if promoted:
    if not (PARENT.is_file() and RUN_TEST.is_file() and RUN_FUTURE.is_file()):
        raise FileNotFoundError(f'Для полного refit нужны {PARENT}, {RUN_TEST}, {RUN_FUTURE}')
    if Path(test_input).resolve()!=RUN_TEST.resolve():
        raise ValueError('Финальный запуск требует test от того же комплекта, что full_history')
    future = pd.read_parquet(RUN_FUTURE,columns=[c for c in COLUMNS if c!='y'])
    future['cutoff'] = pd.to_datetime(future.cutoff)
    request = pd.read_parquet(DATA/'test_solo_track.parquet',columns=['id','route_id','timestamp'])
    parent = pd.read_csv(PARENT)
    if (list(parent.columns)!=['id','y_pred'] or len(parent)!=8000 or len(request)!=8000
            or not parent.id.is_unique or not request.id.is_unique or set(parent.id)!=set(request.id)
            or not np.isfinite(parent.y_pred).all() or parent.y_pred.lt(0).any()):
        raise ValueError('Некорректный родительский сабмит')
    cutoff = request.timestamp.min()-STEP
    if (len(future)!=8000 or future.duplicated(KEYS).any() or not future.cutoff.eq(cutoff).all()
            or TIMES_STATUS[-1]!=cutoff or not np.isfinite(future[[c for c in COLUMNS if c not in KEYS+['y']]].to_numpy()).all()):
        raise ValueError('Некорректные базовые прогнозы на конкурс')
    history = pd.concat([oof,test],ignore_index=True)
    if not (history.cutoff+history.horizon*STEP<=cutoff).all():
        raise ValueError('Обучающие цели за пределами истории')
    print('Полное переобучение метамодели',flush=True)
    matrix = shape_matrix(history)
    weights = fit_weights(matrix,history.y)
    baseline = matrix @ weights
    features,level = design(history,matrix,baseline,True)
    model = CatBoostRegressor(depth=4,iterations=220,loss_function='MAE',learning_rate=.035,
                              l2_leaf_reg=20,random_seed=SEED,verbose=False,allow_writing_files=False,
                              task_type='CPU',thread_count=4)
    model.fit(features,(history.y.to_numpy(dtype=float)-baseline)/level,sample_weight=level/level.mean())
    fitted_final = weights,{'status_meta':model}
    display(pd.DataFrame(summarize(history,predict_pair(history,fitted_final),'full_history','train_oof')))
    future['status_meta'] = predict_pair(future,fitted_final)['status_meta']
    future['timestamp'] = future.cutoff+future.horizon*STEP
    query = request.merge(parent,on='id',validate='one_to_one',sort=False).merge(
        future[['route_id','timestamp','status_meta']],on=['route_id','timestamp'],validate='one_to_one',how='left',sort=False)
    if query.status_meta.isna().any() or not query.id.equals(request.id):
        raise ValueError('Ключи конкурса не совпали')
    query['cutoff'] = cutoff
    query['horizon'] = ((query.timestamp-cutoff)/STEP).astype(int)
    prediction = with_volume(query,query.status_meta.to_numpy(),anchor=query.y_pred.to_numpy())
    destination = OUT/'submission_status_meta_full_history.csv'
    query[['id']].assign(y_pred=prediction).to_csv(destination.with_suffix('.partial'),index=False)
    destination.with_suffix('.partial').replace(destination)
    model.save_model(str(OUT/'status_meta_full_history.cbm'))
    print('Сабмит:',destination,flush=True)
print('Результаты:',OUT,flush=True)
